Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved. SPDX-License-Identifier: Apache-2.0

# Cross-ISV Queries: Salesforce + SAP via Single Gateway

## Overview

This notebook demonstrates the power of routing multiple ISV platforms through a single [Amazon Bedrock AgentCore Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html). With Salesforce (CRM) and SAP (ERP) connected to the same gateway, an AI agent can answer questions that span both systems — without the user needing to know which system holds which data.

**Use cases demonstrated:**
1. Customer 360 — combine SAP business partner data with Salesforce account history
2. Pipeline Reconciliation — compare Salesforce opportunities with SAP sales orders
3. Support Case with ERP Context — enrich Salesforce cases with SAP inventory data
4. Natural Language Agent — let Claude orchestrate cross-system queries autonomously

| Information | Details |
|:---|:---|
| Prerequisites | Both Salesforce + SAP targets READY on same gateway (from notebooks 01 + 02) |
| Tools available | ~52 (43 Salesforce + 9 SAP) |
| Agent framework | Strands Agents |
| LLM | Anthropic Claude Sonnet 4 |

### Prerequisites

This notebook assumes you have completed:
1. [01-salesforce-gateway-target.ipynb](01-salesforce-gateway-target.ipynb) — Salesforce target is READY
2. [02-sap-mcp-server-target.ipynb](02-sap-mcp-server-target.ipynb) — SAP target is READY

Both targets must be on the **same gateway**. You'll provide the gateway connection details below.

In [ ]:
!pip install -r requirements.txt --quiet

In [ ]:
import getpass
import json
import logging
import uuid

import boto3
import requests
from boto3.session import Session

import gateway_mcp_client

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[logging.StreamHandler()],
)

session = Session()
REGION = session.region_name or "us-east-1"
print(f"Using region: {REGION}")

In [ ]:
# Gateway connection details (from notebooks 01 + 02)
GATEWAY_URL = input("Enter Gateway URL: ")
TOKEN_ENDPOINT = input("Enter Cognito token endpoint: ")
GW_CLIENT_ID = input("Enter Cognito Client ID: ")
GW_CLIENT_SECRET = getpass.getpass("Enter Cognito Client Secret: ")
FULL_SCOPE = input("Enter OAuth scope: ")

# Salesforce domain (needed for SF tool calls)
SF_DOMAIN = input("Enter Salesforce domain (e.g., myorg-dev-ed): ")

assert GATEWAY_URL.strip(), "Gateway URL cannot be empty"
assert SF_DOMAIN.strip(), "Salesforce domain cannot be empty"

print(f"\nGateway: {GATEWAY_URL}")
print(f"Salesforce domain: {SF_DOMAIN}")

In [ ]:
# Connect to the gateway
def get_gw_token() -> str:
    return gateway_mcp_client.get_cognito_m2m_token(
        token_endpoint=TOKEN_ENDPOINT,
        client_id=GW_CLIENT_ID,
        client_secret=GW_CLIENT_SECRET,
        scope=FULL_SCOPE,
    )

mcp = gateway_mcp_client.GatewayMCPClient(
    gateway_url=GATEWAY_URL,
    get_token=get_gw_token,
    session_id=str(uuid.uuid4()),
)

# List all tools from both targets
all_tools = mcp.list_all_tools()
sf_tools = [t for t in all_tools if t["name"].startswith("salesforce-target___")]
sap_tools = [t for t in all_tools if t["name"].startswith("sap-target___")]

print(f"Total tools: {len(all_tools)}")
print(f"  Salesforce: {len(sf_tools)}")
print(f"  SAP: {len(sap_tools)}")
print(f"  Other: {len(all_tools) - len(sf_tools) - len(sap_tools)}")

## Use Case 1: Customer 360

Combine customer data from both SAP (business partner master data) and Salesforce (CRM account history) to build a unified view.

**Pattern:** Query SAP for business partner details → Query Salesforce for account + opportunities

In [ ]:
# SAP: Get business partner data
sap_result = mcp.call_tool(
    "sap-target___odata_read",
    {
        "service_name": "API_BUSINESS_PARTNER",
        "entity_set": "A_BusinessPartner",
        "top": 5,
        "select": "BusinessPartner,BusinessPartnerFullName,BusinessPartnerCategory,Industry",
    },
)

print("=== SAP Business Partners ===")
sap_content = sap_result.get("result", {}).get("content", [])
for item in sap_content:
    if item.get("type") == "text":
        try:
            data = json.loads(item["text"])
            print(json.dumps(data, indent=2)[:2000])
        except json.JSONDecodeError:
            print(item["text"][:2000])

In [ ]:
# Salesforce: Get accounts
sf_result = mcp.call_tool(
    "salesforce-target___queryAccounts",
    {
        "domainName": SF_DOMAIN,
        "q": "SELECT Id, Name, Industry, CreatedDate FROM Account LIMIT 5",
    },
)

print("=== Salesforce Accounts ===")
sf_content = sf_result.get("result", {}).get("content", [])
for item in sf_content:
    if item.get("type") == "text":
        try:
            data = json.loads(item["text"])
            print(json.dumps(data, indent=2)[:2000])
        except json.JSONDecodeError:
            print(item["text"][:2000])

## Use Case 2: Pipeline Reconciliation

Compare Salesforce opportunities (sales pipeline) with SAP sales orders to identify which deals have been converted to orders.

**Pattern:** Query Salesforce for open opportunities → Query SAP for matching sales orders

In [ ]:
from strands import Agent
from strands.tools.mcp import MCPClient
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

mcp_client = MCPClient(
    lambda: streamablehttp_client(
        url=GATEWAY_URL,
        headers={
            "Authorization": f"Bearer {get_gw_token()}",
            "MCP-Protocol-Version": "2025-03-26",
        },
    )
)

SYSTEM_PROMPT = (
    "You are a helpful enterprise assistant with access to both Salesforce (CRM) and SAP (ERP) tools "
    "through a unified AgentCore Gateway. "
    f"For Salesforce tools (prefix 'salesforce-target___'), always include domainName='{SF_DOMAIN}'. "
    "Do not pass Content-Type to Salesforce create operations. "
    "For SAP tools (prefix 'sap-target___'), use get_metadata before reads on unfamiliar entity sets. "
    "Use odata_count before odata_read to understand data volume. "
    "When answering cross-system questions, query both systems and synthesize the results."
)

with mcp_client:
    agent = Agent(
        model="us.anthropic.claude-sonnet-4-6-v1",
        system_prompt=SYSTEM_PROMPT,
        tools=mcp_client.list_tools_sync(),
    )

    # Cross-ISV query
    result = agent(
        "Give me a Customer 360 view: get the top 3 business partners from SAP "
        "and all Salesforce accounts, then tell me which customers exist in both systems."
    )
    print(result)

In [ ]:
# SAP: Get recent sales orders
sap_orders = mcp.call_tool(
    "sap-target___odata_read",
    {
        "service_name": "API_SALES_ORDER_SRV",
        "entity_set": "A_SalesOrder",
        "top": 5,
        "select": "SalesOrder,SoldToParty,TotalNetAmount,TransactionCurrency,CreationDate",
    },
)

print("=== SAP Sales Orders ===")
content = sap_orders.get("result", {}).get("content", [])
for item in content:
    if item.get("type") == "text":
        try:
            data = json.loads(item["text"])
            print(json.dumps(data, indent=2)[:2000])
        except json.JSONDecodeError:
            print(item["text"][:2000])

## Use Case 3: Support Case with ERP Context

When creating a support case in Salesforce, enrich it with relevant SAP data (e.g., material stock levels, order history).

**Pattern:** Query SAP for relevant data → Create enriched Salesforce case

In [ ]:
# SAP: Check available services for material/inventory
sap_services = mcp.call_tool(
    "sap-target___find_sap_services",
    {"search_term": "material stock", "top": 3},
)

print("=== SAP Material/Stock Services ===")
content = sap_services.get("result", {}).get("content", [])
for item in content:
    if item.get("type") == "text":
        print(item["text"][:2000])

In [ ]:
# Salesforce: Create a case with ERP context
# Note: Content-Type workaround required for create operations
case_result = mcp.call_tool(
    "salesforce-target___createCase",
    {
        "domainName": SF_DOMAIN,
        "Content-Type": "",
        "Subject": "Stock Discrepancy - Cross-System Investigation",
        "Description": "Customer reported inventory mismatch. SAP material stock checked via AgentCore Gateway.",
        "Status": "New",
        "Priority": "Medium",
        "Origin": "Web",
    },
)

print("=== Created Salesforce Case ===")
content = case_result.get("result", {}).get("content", [])
for item in content:
    if item.get("type") == "text":
        print(item["text"][:1000])

## Use Case 4: Natural Language Agent (Cross-ISV)

The most powerful pattern: let a Strands Agent handle cross-system queries autonomously. The agent has access to all 52+ tools and decides which systems to query based on the natural language request.

In [ ]:
from strands import Agent
from strands.tools.mcp import MCPClient
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

mcp_client = MCPClient(
    lambda: streamablehttp_client(
        url=GATEWAY_URL,
        headers={
            "Authorization": f"Bearer {get_gw_token()}",
            "MCP-Protocol-Version": "2025-11-25",
        },
    )
)

SYSTEM_PROMPT = (
    "You are a helpful enterprise assistant with access to both Salesforce (CRM) and SAP (ERP) tools "
    "through a unified AgentCore Gateway. "
    f"For Salesforce tools (prefix 'salesforce-target___'), always include domainName='{SF_DOMAIN}'. "
    "Do not pass Content-Type to Salesforce create operations. "
    "For SAP tools (prefix 'sap-target___'), use get_metadata before reads on unfamiliar entity sets. "
    "Use odata_count before odata_read to understand data volume. "
    "When answering cross-system questions, query both systems and synthesize the results."
)

with mcp_client:
    agent = Agent(
        model="us.anthropic.claude-sonnet-4-6-v1",
        system_prompt=SYSTEM_PROMPT,
        tools=mcp_client.list_tools_sync(),
    )

    # Cross-ISV query
    result = agent(
        "Give me a Customer 360 view: get the top 3 business partners from SAP "
        "and all Salesforce accounts, then tell me which customers exist in both systems."
    )
    print(result)

In [ ]:
# Another cross-ISV query: pipeline reconciliation
with mcp_client:
    agent = Agent(
        model="us.anthropic.claude-sonnet-4-6-v1",
        system_prompt=SYSTEM_PROMPT,
        tools=mcp_client.list_tools_sync(),
    )

    result = agent(
        "Compare the Salesforce pipeline with SAP orders: "
        "get open Salesforce opportunities and recent SAP sales orders, "
        "then identify any patterns or gaps between the two systems."
    )
    print(result)

## Summary

| Use Case | Salesforce Tools | SAP Tools | Value |
|---|---|---|---|
| Customer 360 | queryAccounts, getAccountById | odata_read (A_BusinessPartner) | Unified customer view across CRM + ERP |
| Pipeline Reconciliation | queryAccounts (Opportunities) | odata_read (A_SalesOrder) | Track deal-to-order conversion |
| Support + ERP Context | createCase | find_sap_services, odata_read | Enriched support tickets |
| Natural Language Agent | All 43 SF tools | All 9 SAP tools | AI-driven cross-system intelligence |

**Key takeaway:** A single AgentCore Gateway endpoint consolidates access to multiple ISV platforms, enabling AI agents to answer cross-system questions without custom integration code.

## Clean Up

If you're done with all three notebooks, clean up the gateway and related resources. Refer to the cleanup cells in [01-salesforce-gateway-target.ipynb](01-salesforce-gateway-target.ipynb) and [02-sap-mcp-server-target.ipynb](02-sap-mcp-server-target.ipynb) for the full deletion sequence:

1. Delete gateway targets (both Salesforce and SAP)
2. Delete credential providers
3. Delete the gateway
4. Delete Cognito resources (domain, then user pool)
5. Delete IAM role